# 4.1 Planning with a Graph: Does a DAG of Sub-Questions Beat Step-by-Step ReAct on Multi-Hop Questions?

*Formerly notebook 25.*

> **Outputs in this copy come from the confidence run** (`SMASH_MODE=batch`): MuSiQue 1,000, 2Wiki 1,000, MetaQA 3-hop
> 300 (latency on 200), 31,788 calls through the Batch API, $1.85. Task 4.1. They are reported in `RESULTS-REPORT.md`
> and the v2 paper.

**Presentation map:** Task 4.1 (planning with a graph): a task decomposed into a dependency graph of sub-tasks,
independent ones run in parallel (LLMCompiler, Kim et al., ICML 2024), against a step-by-step agent loop (ReAct).

## Frozen protocol (PLAN §5, task 4.1; written before the test run)

**Research question.** Does planning a DAG of sub-questions beat step-by-step ReAct on multi-hop questions, in
accuracy, LLM calls and latency?

**Hypothesis.** LLMCompiler reports equal accuracy to ReAct on HotpotQA with 1.8–3.7× lower latency and 3–7× lower
cost, on benchmarks that favour parallel calls. On sequential 2–4-hop chains (MuSiQue, MetaQA) parallelism has little
to offer, so we expect the DAG planner to match ReAct's accuracy within a few F1 points, to use fewer tokens (no
growing transcript), and to be faster mainly on 2Wiki's comparison questions, whose sub-questions are independent.

**Data** (all local, no download):
- **MuSiQue** (answerable, dev) and **2WikiMultiHopQA** (dev), 1,000 questions each, over Notebook 1.4a's passage
  pools (11,656 and 6,119 Wikipedia paragraphs) and its `text-embedding-3-small` passage index, read from its cache
  by name (nothing is re-embedded). MuSiQue labels a gold `question_decomposition` (2 hops: 518, 3: 316, 4: 166).
- **MetaQA 3-hop**, Notebook 1.2b's sample of 300 questions (seed 7) over its index of the 18,127 WikiMovies articles.

**Conditions** (`gpt-6-luna`, no reasoning, temperature 0; the same `search` tool everywhere: the top 5 passages by
cosine similarity in the benchmark's index):
1. **Vector RAG (single shot):** Notebook 1.4a's answers on the same index (top 5) and Notebook 1.2b's (top 10) for
   MetaQA, reused from their caches where the request matches exactly, otherwise recomputed under this notebook's stage.
2. **ReAct:** a tool-calling loop with one `search` per step (parallel tool calls disabled), at most 5 searches,
   then an answer without tools. Every step is one Batch job over all episodes.
3. **List decomposition:** one call writes an ordered list of at most 5 sub-questions (`#k` refers to the answer of
   sub-question k); they are answered **one after another**, each by one search and one short-answer call that sees
   all earlier answers; a final call answers the question from the sub-answers and their top passages.
4. **DAG planner (LLMCompiler-style):** one call writes at most 5 sub-questions with explicit dependencies; nodes
   whose dependencies are answered run **in parallel** (one wave = one Batch job); each node sees only its
   dependencies' answers; a join call answers from the sub-answers and their top passages (same join prompt as 3).

Conditions 3 and 4 use the same planning instructions, sub-answer prompt and final prompt; they differ only in the
plan's structure (list vs. graph) and therefore in what runs in parallel and what context each node sees.

**Primary metric:** answer accuracy, token F1 on MuSiQue and 2Wiki (Notebook 1.4a's scorer), Hits@1 on MetaQA
(Notebook 1.2b's scorer). **Primary contrast:** DAG − ReAct, paired 95% bootstrap interval over questions, per
benchmark and pooled over MuSiQue + 2Wiki (n = 2,000). The verdict is decided on this contrast only.

**Secondary metrics:** EM; LLM calls, input/output tokens and USD per question; sequential LLM rounds per question
(the critical path, a latency proxy for all questions); **wall-clock latency** on a synchronous subset (the first
100 MuSiQue + 100 2Wiki questions, each question run alone, standard tier); for MuSiQue, how often the planned DAG
(and list) has the gold number of hops and the gold dependency structure. Results by hop count (MuSiQue 2/3/4 hops;
2Wiki question type; MetaQA 3 hops). Also DAG − list and DAG − vector RAG.

**n:** 1,000 + 1,000 + 300 = 2,300 questions per condition; latency subset 200. Test run: the first 25 per benchmark
(fixed order) and 5 + 5 latency questions.

**Threats.** (i) Prompts are fixed before the run and not tuned on test items; a better prompt for either side could
change the result. (ii) ReAct is limited to one search per step: this is classic ReAct and LLMCompiler's baseline,
but a function-calling agent allowed parallel calls would sit between ReAct and the DAG. (iii) A sub-question answered
`unknown` or wrongly propagates through the plan; ReAct can re-search. (iv) MetaQA's intermediate answers are sets
(e.g. all films of a director); one search of 5 articles per sub-question cannot cover them, which caps all methods.
(v) Latency is measured once per question on the public API at standard tier and depends on its load; the
sequential-rounds count is the load-independent proxy. (vi) In test mode Notebook 1.4a's passage pool is restricted to
the paragraphs of the 25 test questions (as in Notebooks 1.4a and 1.5), so retrieval is easier than in the full run.
(vii) MetaQA's vector baseline (Notebook 1.2b) reads 10 articles, the agents' tool returns 5 per search.

**Protocol changes after the test run:** none yet.

In [ ]:
# Notebooks live in examples/<task group>/; data/, smash_batch.py and the .venv-* environments stay in examples/.
# Every path below is relative to examples/, so run from there whether the notebook is opened in its own folder or not.
import os, sys
if not os.path.exists("smash_batch.py") and os.path.exists(os.path.join("..", "smash_batch.py")):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [1]:
import json, os, re, string, time, random, unicodedata
from collections import Counter, defaultdict
import numpy as np
import networkx as nx
from pydantic import BaseModel, Field
from smash_batch import LLM, mode_from_env, wilson, paired_boot, PRICE, EMBED_MODEL, EMBED_PRICE

MODE = mode_from_env()               # replay (default) | test (small live run) | batch (full run via Batch API)
TEST = MODE == "test"
BASE, MQA = "data/benchmarks/musique_2wiki", "data/benchmarks/metaqa"
N_Q = 25 if TEST else 1000           # MuSiQue / 2Wiki questions (test: the first 25 of the same fixed order)
N_META = 25 if TEST else 300         # MetaQA 3-hop questions (test: the first 25 of Notebook 1.2b's sample of 300)
N_LAT = 5 if TEST else 100           # latency subset per benchmark (MuSiQue, 2Wiki): the first N_LAT questions
K, MAX_STEPS, MAX_NODES = 5, 5, 5    # passages per search; ReAct searches; sub-questions per plan

llm = LLM(f"{BASE}/cache_25" + ("_test" if TEST else ""))                 # this notebook's own cache (stages nb25_*)
nb11 = LLM(f"{BASE}/cache" + ("_test" if TEST else ""), mode="replay")    # Notebook 1.4a's cache, read only
nb19 = LLM(f"{MQA}/cache_19", mode="replay")                              # Notebook 1.2b's cache, read only

# --- MuSiQue and 2Wiki: Notebook 1.4a's loader, unchanged ---
DATA = {}
for name in ("musique", "2wikimultihopqa"):
    qs = json.load(open(f"{BASE}/{name}.json"))
    corpus = json.load(open(f"{BASE}/{name}_corpus.json"))
    for q in qs:
        if name == "musique":
            q["qid"], q["golds"] = q["id"], [q["answer"]] + q.get("answer_aliases", [])
            q["group"] = q["id"].split("hop")[0] + "-hop"
        else:
            q["qid"], q["golds"] = q["_id"], [q["answer"]]
            q["group"] = q["type"]
    DATA[name] = {"questions": qs[:N_Q], "corpus": corpus}
    print(f"{name}: {len(qs):,} questions, {len(corpus):,} passages | groups:", dict(Counter(q["group"] for q in qs)))
print("MODE =", MODE)

musique: 1,000 questions, 11,656 passages | groups: {'2-hop': 518, '3-hop': 316, '4-hop': 166}
2wikimultihopqa: 1,000 questions, 6,119 passages | groups: {'compositional': 413, 'comparison': 244, 'bridge_comparison': 235, 'inference': 108}
MODE = batch


## The indexes of Notebooks 1.4a and 1.2b (read from their caches)

The passage pools and question lists are rebuilt exactly as in Notebook 1.4a (cell 3) and Notebook 1.2b (cell 1), so the
embeddings are found in their caches by name and hash; nothing is re-embedded. MetaQA always uses Notebook 1.2b's full
index (also in test mode), because its test cache holds a different sample.

In [2]:
for name, d in DATA.items():
    passages = [p["title"] + "\n" + p["text"] for p in d["corpus"]]
    if TEST:                                     # Notebook 1.4a's test pool: only the paragraphs of the test questions
        keep = set()
        for q in d["questions"]:
            if name == "musique":
                keep |= {p["title"] + "\n" + p["paragraph_text"] for p in q["paragraphs"]}
            else:
                keep |= {t + "\n" + " ".join(s) for t, s in q["context"]}
        passages = [p for p in passages if p in keep]
    d["passages"] = passages
    d["P"] = nb11.embed(f"{name}_passages", passages)
    d["Q"] = nb11.embed(f"{name}_questions", [q["question"] for q in d["questions"]])
    print(f"{name}: {len(passages):,} passages, index {d['P'].shape}, {len(d['questions'])} questions")

# --- MetaQA: Notebook 1.2b's articles, questions and sample (seed 7, 300 per hop), unchanged ---
def key(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", s)).strip()

movies_kb = set()
for line in open(f"{MQA}/kb.txt", encoding="utf-8"):
    movies_kb.add(line.split("|", 1)[0])
articles, cur = [], None
for line in open(f"{MQA}/wikimovies_text/movieqa/knowledge_source/wiki.txt", encoding="utf-8"):
    line = line.rstrip("\n")
    if not line:
        cur = None; continue
    n, txt = line.split(" ", 1)
    if n == "1":
        cur = {"title": txt, "lines": []}; articles.append(cur)
    elif cur is not None:
        cur["lines"].append(txt)
strip_title = lambda t: re.sub(r"\s*\((?:[^()]*\bfilm|[^()]*\bmovie)\)$", "", t)
for i, a in enumerate(articles):
    a["movie"], a["text"] = strip_title(a["title"]), " ".join(a["lines"])
movie_keys = {key(a["movie"]) for a in articles}

def load_split(hop):
    rows = []
    for i, line in enumerate(open(f"{MQA}/qa_test_{hop}hop.txt", encoding="utf-8")):
        q, a = line.rstrip("\n").split("\t")
        rows.append({"qid": f"{hop}hop-{i}", "hop": hop, "question": q,
                     "topic": re.search(r"\[(.+?)\]", q).group(1), "gold": a.split("|")})
    return rows
splits = {h: load_split(h) for h in (1, 2, 3)}
eligible = {h: [q for q in s if q["topic"] in movies_kb and key(q["topic"]) in movie_keys] for h, s in splits.items()}
rng = random.Random(7)
Q19 = [q for h in (1, 2, 3) for q in rng.sample(eligible[h], 300)]          # Notebook 1.2b's full sample (900)
A_EMB = nb19.embed("articles", [f"{a['title']}\n{a['text']}" for a in articles], cap=2000)
Q19_EMB = nb19.embed("questions", [q["question"] for q in Q19])
meta_idx = [i for i, q in enumerate(Q19) if q["hop"] == 3][:N_META]
metaqa = [dict(Q19[i], golds=Q19[i]["gold"], group="3-hop") for i in meta_idx]
DATA["metaqa"] = {"questions": metaqa, "passages": [f"{a['title']}: {a['text']}" for a in articles],
                  "P": A_EMB, "Q": Q19_EMB[meta_idx]}
print(f"metaqa: {len(articles):,} articles, index {A_EMB.shape}, {len(metaqa)} 3-hop questions of Notebook 1.2b's sample")
BENCH = list(DATA)

musique: 11,656 passages, index (11656, 1536), 1000 questions
2wikimultihopqa: 6,119 passages, index (6119, 1536), 1000 questions


metaqa: 18,127 articles, index (18127, 1536), 300 3-hop questions of Notebook 1.2b's sample


## The shared search tool and answer rules

`search(query)` embeds the query with the same model as the index and returns the 5 most similar passages. Query
embeddings are cached per stage under this notebook's names. MuSiQue and 2Wiki answers end with `Answer: <short
answer>` (Notebook 1.4a's rule); MetaQA answers are a JSON list of entities (Notebook 1.2b's rule), scored by Hits@1.

In [3]:
CORPUS = {"musique": "a collection of Wikipedia paragraphs", "2wikimultihopqa": "a collection of Wikipedia paragraphs",
          "metaqa": "a collection of Wikipedia articles about movies (first paragraph each)"}
TEXT_RULE = ("Reason step by step in at most three short sentences, then give the final answer on the last line as "
             "'Answer: <answer>'. The answer must be a short phrase (an entity, date, number or yes/no), not a sentence.")
JSON_RULE = ("Return JSON {\"answers\": [...]} listing EVERY entity that answers the question, using the exact "
             "entity names; put the most likely answer first; do not include the topic entity itself unless it is "
             "an answer; return an empty list if you cannot find the answer.")   # Notebook 1.2b's ANSWER_RULES
RULE = {b: JSON_RULE if b == "metaqa" else TEXT_RULE for b in BENCH}

def body(bench, msgs, max_tokens=400, tools=None, json_out=False):
    b = LLM.body(msgs, max_tokens=max_tokens, tools=tools)
    if json_out:
        b["response_format"] = {"type": "json_object"}
    if tools:
        b["parallel_tool_calls"] = False          # ReAct: one action per step
    return b

class Runner:
    """Sends chat requests and embeds queries. The main runner uses the cache and Batch; the latency runner
    (further below) sends one question at a time, synchronously."""
    def __init__(self, llm, prefix):
        self.llm, self.prefix = llm, prefix
    def chat(self, stage, reqs, parse=None):
        return self.llm.run(f"{self.prefix}{stage}", reqs, parse=parse) if reqs else {}
    def embed(self, name, texts):
        return self.llm.embed(f"{self.prefix}{name}", texts) if texts else np.zeros((0, 1536), np.float32)

def search(R, stage, items):
    """items: [(bench, query)] -> [list of K passage ids], one embedding call per benchmark."""
    out = [None] * len(items)
    for b in BENCH:
        idx = [i for i, (bb, _) in enumerate(items) if bb == b]
        if not idx:
            continue
        E = R.embed(f"{stage}_{b}", [items[i][1] for i in idx])
        for i, e in zip(idx, E):
            out[i] = [int(j) for j in np.argsort(-(DATA[b]["P"] @ e))[:K]]
    return out

def show(b, ids, start=1):
    return "\n\n".join(f"[{n}] {DATA[b]['passages'][i]}" for n, i in enumerate(ids, start))

def new_ep(b, qi):
    q = DATA[b]["questions"][qi]
    return {"bench": b, "qi": qi, "qid": q["qid"], "question": q["question"], "calls": 0, "tin": 0, "tout": 0,
            "rounds": 0, "text": None}

def acc(ep, r):
    ep["calls"] += 1; ep["tin"] += r.get("in", 0); ep["tout"] += r.get("out", 0)

ek = lambda ep: f"{ep['bench']}|{ep['qid']}"

## Condition 1: single-shot vector RAG (reused from Notebooks 1.4a and 1.2b)

The request bodies are rebuilt exactly as those notebooks built them and looked up in their caches by key and hash.
Any request that does not match is recomputed under this notebook's stage `nb25_vector`.

In [4]:
NB11_SYS = ("Answer the question using the passages. Reason step by step in at most three short sentences, then "
            "give the final answer on the last line as 'Answer: <answer>'. The answer must be a short phrase "
            "(an entity, date, number or yes/no), not a sentence.")

def vector_body(b, qi):
    d = DATA[b]; q = d["questions"][qi]
    if b == "metaqa":                            # Notebook 1.2b, cell 7
        top = np.argsort(-(A_EMB @ d["Q"][qi]))[:10]
        msgs = [{"role": "system", "content": "Answer movie questions using ONLY the retrieved documents. " + JSON_RULE},
                {"role": "user", "content": f"Question: {q['question']}\n\nRetrieved documents:\n" +
                 "\n".join(f"[{j + 1}] {articles[i]['title']}: {articles[i]['text']}" for j, i in enumerate(top))}]
        bd = LLM.body(msgs, max_tokens=1500); bd["response_format"] = {"type": "json_object"}
        return q["qid"], bd
    ids = list(np.argsort(-(d["P"] @ d["Q"][qi]))[:5])          # Notebook 1.4a, cells 10 and 12
    ctx = "\n\n".join(f"[{n + 1}] {d['passages'][i]}" for n, i in enumerate(ids))
    return f"vector|{q['qid']}", LLM.body([{"role": "system", "content": NB11_SYS},
                                          {"role": "user", "content": f"{ctx}\n\nQuestion: {q['question']}"}], max_tokens=300)

def borrow(src, stage, reqs):
    rows = src.rows(stage)
    hit = {k: rows[k]["r"] for k, b in reqs.items()
           if k in rows and rows[k]["h"] == LLM._hash(b) and "error" not in rows[k]["r"]}
    return hit, {k: b for k, b in reqs.items() if k not in hit}

VEC, VEC_SRC = {}, Counter()
for b in BENCH:
    reqs = dict(vector_body(b, qi) for qi in range(len(DATA[b]["questions"])))
    src, stage = (nb19, "vector") if b == "metaqa" else (nb11, f"{b}_answers")
    hit, miss = borrow(src, stage, reqs)
    new = llm.run("nb25_vector", {f"{b}|{k}": v for k, v in miss.items()}) if miss else {}
    for qi, q in enumerate(DATA[b]["questions"]):
        k = vector_body(b, qi)[0]
        VEC[(b, q["qid"])] = hit[k] if k in hit else new[f"{b}|{k}"]
    VEC_SRC[b] = (len(hit), len(miss))
    print(f"{b}: {len(hit)} vector answers reused from Notebook {19 if b == 'metaqa' else 11}, {len(miss)} recomputed")

musique: 1000 vector answers reused from Notebook 1.4a, 0 recomputed


2wikimultihopqa: 1000 vector answers reused from Notebook 1.4a, 0 recomputed


metaqa: 300 vector answers reused from Notebook 1.2b, 0 recomputed


## Condition 2: ReAct with the same search tool

One `search` per step (parallel tool calls off), at most 5 searches; at step 6 the tools are removed and the agent
must answer. Each step is one call over all active episodes (one Batch job in batch mode).

In [5]:
SEARCH_TOOL = [{"type": "function", "function": {
    "name": "search",
    "description": f"Search the collection; returns the {K} passages most similar to the query.",
    "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "a search query"}},
                   "required": ["query"]}}}]

def react_sys(b):
    return (f"You answer a question by searching {CORPUS[b]} with the search tool, which returns the {K} passages "
            "most similar to a query. Search for one thing at a time: decide which fact you need next, search for "
            f"it, read the results, and repeat. You can search at most {MAX_STEPS} times. When you have enough "
            "information, stop searching and answer. " + RULE[b])

def run_react(R, eps, tag="react"):
    for ep in eps:
        ep["msgs"] = [{"role": "system", "content": react_sys(ep["bench"])}, {"role": "user", "content": ep["question"]}]
        ep["searches"] = 0
    for step in range(1, MAX_STEPS + 2):
        active = [ep for ep in eps if ep["text"] is None]
        if not active:
            break
        tools = SEARCH_TOOL if step <= MAX_STEPS else None
        res = R.chat(f"{tag}_s{step}", {ek(ep): body(ep["bench"], ep["msgs"], 400, tools, ep["bench"] == "metaqa")
                                         for ep in active})
        items, where = [], []
        for ep in active:
            r = res[ek(ep)]; acc(ep, r); ep["rounds"] += 1
            if not r.get("tool_calls") or step > MAX_STEPS:
                ep["text"] = r.get("text") or ""; continue
            ep["msgs"].append({"role": "assistant", "content": r["text"] or None, "tool_calls": [
                {"id": t["id"], "type": "function", "function": {"name": t["name"], "arguments": t["arguments"]}}
                for t in r["tool_calls"]]})
            for t in r["tool_calls"]:
                try:
                    qy = str(json.loads(t["arguments"]).get("query", ""))
                except Exception:
                    qy = ""
                items.append((ep["bench"], qy or ep["question"])); where.append((ep, t["id"]))
        for (ep, tid), ids in zip(where, search(R, f"{tag}_s{step}", items)):
            ep["searches"] += 1
            ep["msgs"].append({"role": "tool", "tool_call_id": tid, "content": show(ep["bench"], ids)})
    return eps

REACT = {(ep["bench"], ep["qid"]): ep for ep in
         run_react(Runner(llm, "nb25_"), [new_ep(b, qi) for b in BENCH for qi in range(len(DATA[b]["questions"]))])}
print("ReAct: mean searches per question", {b: round(np.mean([e["searches"] for (bb, _), e in REACT.items() if bb == b]), 2)
                                           for b in BENCH})

ReAct: mean searches per question {'musique': np.float64(2.83), '2wikimultihopqa': np.float64(2.15), 'metaqa': np.float64(4.99)}


## Conditions 3 and 4: list decomposition and the DAG planner

Both planners get the same instructions about what a sub-question is; the DAG planner additionally states the
dependencies and is told that independent sub-questions run in parallel. Execution is shared: a node's `#k` is
replaced by the answer of node k, the node runs one search and one short-answer call. In the **list**, node i runs in
round i and sees all earlier answers; in the **DAG**, a node runs as soon as its dependencies are answered (its wave
= 1 + the longest dependency chain) and sees only its dependencies' answers. Each wave is one call over all
episodes. The final/join call sees the sub-questions, their answers and the top 3 passages of each search.

In [6]:
class SubQ(BaseModel):
    id: int
    question: str
    depends_on: list[int] = Field(description="ids of the sub-questions whose answers this one needs")

class DagPlan(BaseModel):
    sub_questions: list[SubQ]

class ListPlan(BaseModel):
    sub_questions: list[str]

PLAN_COMMON = ("Break the question into the sub-questions needed to answer it. Each sub-question must be answerable "
               "by one search of {corpus}: it asks for one fact (or one set of entities) about one named entity. "
               "Refer to the answer of an earlier sub-question k as #k. Use at most " + str(MAX_NODES) + " "
               "sub-questions and no more than needed. Do not add a sub-question for the final comparison or "
               "combination of the answers; that is done after all sub-questions are answered. ")
LIST_SYS = PLAN_COMMON + "Return the sub-questions as a list, in the order in which they must be answered."
DAG_SYS = PLAN_COMMON + ("Return them as a dependency graph: number them 1, 2, ...; for each, list in depends_on the "
                         "ids of the sub-questions whose answers it uses (exactly those it refers to as #k). "
                         "Sub-questions that do not depend on each other are answered in parallel, so do not make one "
                         "depend on another unless it needs that answer.")
SUB_SYS = ("Answer the sub-question using only the passages and the answers found so far. Reply with the answer only: "
           "the shortest phrase that answers it (an entity, date, number or yes/no) or, if the sub-question asks for "
           "several entities, a list separated by '; ' (at most 10). Reply 'unknown' if the passages do not say.")
JOIN_SYS = "Answer the question using the sub-questions answered so far and the passages they were based on. "
REF = re.compile(r"#(\d+)")

def norm_plan(kind, parsed, question):
    """-> list of nodes {q, deps (0-based, earlier only), refs {k: idx}}; a plan that did not parse is the question."""
    if kind == "list":
        qs = [s for s in (parsed or {}).get("sub_questions", []) if s.strip()][:MAX_NODES]
        nodes = [{"q": s, "ids": i + 1, "decl": list(range(1, i + 1))} for i, s in enumerate(qs)]
    else:
        sq = [s for s in (parsed or {}).get("sub_questions", []) if s["question"].strip()][:MAX_NODES]
        nodes = [{"q": s["question"], "ids": s["id"], "decl": s["depends_on"]} for s in sq]
    if not nodes:
        return [{"q": question, "deps": [], "refs": {}}]
    pos = {}
    for i, n in enumerate(nodes):
        pos.setdefault(n["ids"], i)
    out = []
    for i, n in enumerate(nodes):
        refs = {int(k): pos[int(k)] for k in REF.findall(n["q"]) if int(k) in pos and pos[int(k)] < i}
        deps = sorted({pos[k] for k in n["decl"] if k in pos and pos[k] < i} | set(refs.values()))
        out.append({"q": n["q"], "deps": deps, "refs": refs})
    return out

def fill(ep, node):
    # Replace #k by the answer of node k ('unknown' answers keep the sub-question text instead).
    def rep(m):
        k = int(m.group(1))
        if k not in node["refs"]:
            return m.group(0)
        src = ep["nodes"][node["refs"][k]]
        return src["a"] if src.get("a") and src["a"].lower() != "unknown" else f"(the answer to: {src['qf']})"
    return REF.sub(rep, node["q"])

def run_planner(R, eps, kind, tag):
    reqs = {ek(ep): body(ep["bench"], [{"role": "system", "content": (DAG_SYS if kind == "dag" else LIST_SYS)
                                        .format(corpus=CORPUS[ep["bench"]])},
                                       {"role": "user", "content": f"Question: {ep['question']}"}], 600)
            for ep in eps}
    for ep in eps:
        reqs[ek(ep)]["response_format"] = LLM.body([], schema=DagPlan if kind == "dag" else ListPlan)["response_format"]
    res = R.chat(f"{tag}_plan", reqs, parse=DagPlan if kind == "dag" else ListPlan)
    for ep in eps:
        r = res[ek(ep)]; acc(ep, r); ep["rounds"] += 1
        ep["plan_ok"] = r["parsed"] is not None
        ep["nodes"] = norm_plan(kind, r["parsed"], ep["question"])
        for i, n in enumerate(ep["nodes"]):     # list: one node per round; DAG: 1 + longest dependency chain
            n["wave"] = i + 1 if kind == "list" else 1 + max((ep["nodes"][j]["wave"] for j in n["deps"]), default=0)
            n["ctx"] = list(range(i)) if kind == "list" else n["deps"]
    for w in range(1, MAX_NODES + 1):
        todo = [(ep, i) for ep in eps for i, n in enumerate(ep["nodes"]) if n["wave"] == w]
        if not todo:
            continue
        for ep, i in todo:
            ep["nodes"][i]["qf"] = fill(ep, ep["nodes"][i])
        hits = search(R, f"{tag}_w{w}", [(ep["bench"], ep["nodes"][i]["qf"]) for ep, i in todo])
        reqs = {}
        for (ep, i), ids in zip(todo, hits):
            n = ep["nodes"][i]; n["ids"] = ids
            known = "\n".join(f"#{j + 1}. {ep['nodes'][j]['qf']} -> {ep['nodes'][j]['a']}" for j in n["ctx"])
            msg = (f"Passages:\n{show(ep['bench'], ids)}\n\nMain question: {ep['question']}\n" +
                   (f"Answers found so far:\n{known}\n" if known else "") + f"Sub-question: {n['qf']}")
            reqs[f"{ek(ep)}|{i}"] = body(ep["bench"], [{"role": "system", "content": SUB_SYS},
                                                       {"role": "user", "content": msg}], 80)
        res = R.chat(f"{tag}_w{w}", reqs)
        for ep, i in todo:
            r = res[f"{ek(ep)}|{i}"]; acc(ep, r)
            txt = (r.get("text") or "").strip()
            ep["nodes"][i]["a"] = txt.splitlines()[0].strip() if txt else "unknown"
        for ep in {id(ep): ep for ep, _ in todo}.values():
            ep["rounds"] += 1
    reqs = {}
    for ep in eps:
        sub = "\n".join(f"{j + 1}. {n['qf']} -> {n['a']}" for j, n in enumerate(ep["nodes"]))
        pids = list(dict.fromkeys(p for n in ep["nodes"] for p in n["ids"][:3]))
        reqs[ek(ep)] = body(ep["bench"], [{"role": "system", "content": JOIN_SYS + RULE[ep["bench"]]},
                                          {"role": "user", "content": f"Sub-questions and answers:\n{sub}\n\nPassages:\n"
                                           f"{show(ep['bench'], pids)}\n\nQuestion: {ep['question']}"}],
                            1500 if ep["bench"] == "metaqa" else 300, json_out=ep["bench"] == "metaqa")
    res = R.chat(f"{tag}_final", reqs)
    for ep in eps:
        r = res[ek(ep)]; acc(ep, r); ep["rounds"] += 1
        ep["text"] = r.get("text") or ""
    return eps

ALL_EPS = lambda: [new_ep(b, qi) for b in BENCH for qi in range(len(DATA[b]["questions"]))]
LIST = {(e["bench"], e["qid"]): e for e in run_planner(Runner(llm, "nb25_"), ALL_EPS(), "list", "list")}
DAG = {(e["bench"], e["qid"]): e for e in run_planner(Runner(llm, "nb25_"), ALL_EPS(), "dag", "dag")}
for nm, M in (("list", LIST), ("DAG", DAG)):
    print(f"{nm}: plans parsed {np.mean([e['plan_ok'] for e in M.values()]):.0%}; sub-questions per plan " +
          ", ".join(f"{b} {np.mean([len(e['nodes']) for (bb, _), e in M.items() if bb == b]):.2f}" for b in BENCH) +
          "; sequential rounds " + ", ".join(f"{b} {np.mean([e['rounds'] for (bb, _), e in M.items() if bb == b]):.2f}" for b in BENCH))

ex = DAG[("musique", DATA["musique"]["questions"][0]["qid"])]
print("\nExample DAG:", ex["question"])
for j, n in enumerate(ex["nodes"]):
    print(f"  {j + 1}. [wave {n['wave']}, needs {[d + 1 for d in n['deps']]}] {n['qf']}  ->  {n['a']}")
print("  join:", ex["text"].strip().splitlines()[-1] if ex["text"].strip() else "")

[nb25_dag_w3] 364 new requests -> batch (0 cached)


[nb25_dag_w3] resuming: 1 running, 0 queued


[nb25_dag_w3] batch batch_6aeef56d310c81909b83862785ed1090 completed: 364 ok, 0 failed


[nb25_dag_w3] 1 min: 0 running, 0 queued, 364 results cached


[nb25_dag_w3] done in 0.5 min


[nb25_dag_w4] 52 new requests -> batch (0 cached)


... (129 outputs left out)

[nb25_dag_final] 203 min: 1 running, 0 queued, 0 results cached


[nb25_dag_final] batch batch_6af0d84ebd3c8190ab0930816b734255 completed: 2,300 ok, 0 failed


[nb25_dag_final] done in 205.8 min


list: plans parsed 100%; sub-questions per plan musique 2.54, 2wikimultihopqa 2.36, metaqa 2.32; sequential rounds musique 4.54, 2wikimultihopqa 4.36, metaqa 4.32
DAG: plans parsed 100%; sub-questions per plan musique 2.45, 2wikimultihopqa 2.33, metaqa 2.43; sequential rounds musique 4.16, 2wikimultihopqa 3.65, metaqa 4.36

Example DAG: When was the person who Messi's goals in Copa del Rey compared to get signed by Barcelona?
  1. [wave 1, needs []] Which person’s goals in the Copa del Rey were compared to Lionel Messi’s?  ->  Diego Maradona
  2. [wave 2, needs [1]] When was Diego Maradona signed by FC Barcelona?  ->  June 1982
  join: June 1982


## Scoring

MuSiQue and 2Wiki: the `Answer:` line (or the last line), EM and token F1 against the gold answer and its aliases
(Notebook 1.4a's functions). MetaQA: the JSON list, Hits@1 and set F1 (Notebook 1.2b's functions).

In [7]:
def final(text):
    m = re.findall(r"Answer:\s*(.+)", text)
    return (m[-1] if m else text.strip().splitlines()[-1] if text.strip() else "").strip()

def normalize(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1(pred, gold):
    p, g = normalize(pred).split(), normalize(gold).split()
    common = Counter(p) & Counter(g); same = sum(common.values())
    if same == 0:
        return 0.0
    pr, rc = same / len(p), same / len(g)
    return 2 * pr * rc / (pr + rc)

def parse_answers(text):
    try:
        return [str(a) for a in json.loads(text).get("answers", [])]
    except Exception:
        return []

def score(b, q, text):
    if b == "metaqa":
        P, Gs = [key(p) for p in parse_answers(text)], {key(g) for g in q["golds"]}
        tp = len(set(P) & Gs)
        prec, rec = (tp / len(set(P)) if P else 0.0), tp / len(Gs)
        h1 = float(bool(P) and P[0] in Gs)
        return {"acc": h1, "em": h1, "f1": 2 * prec * rec / (prec + rec) if tp else 0.0}
    pred = final(text)
    s = {"em": float(max(normalize(pred) == normalize(g) for g in q["golds"])), "f1": max(f1(pred, g) for g in q["golds"])}
    s["acc"] = s["f1"]
    return s

METHODS = ("vector", "react", "list", "dag")
NAMES = {"vector": "vector RAG", "react": "ReAct", "list": "list decomp.", "dag": "DAG planner"}
SC = []
for b in BENCH:
    for q in DATA[b]["questions"]:
        k = (b, q["qid"])
        v = VEC[k]
        row = {"bench": b, "qid": q["qid"], "group": q["group"],
               "vector": {**score(b, q, v["text"]), "calls": 1, "tin": v["in"], "tout": v["out"], "rounds": 1}}
        for m, M in (("react", REACT), ("list", LIST), ("dag", DAG)):
            e = M[k]
            row[m] = {**score(b, q, e["text"]), "calls": e["calls"], "tin": e["tin"], "tout": e["tout"], "rounds": e["rounds"]}
        for m in METHODS:
            row[m]["usd"] = (row[m]["tin"] * PRICE["batch"][0] + row[m]["tout"] * PRICE["batch"][1]) / 1e6
        SC.append(row)
print(f"scored {len(SC)} questions x {len(METHODS)} conditions")

scored 2300 questions x 4 conditions

## Accuracy with 95% confidence intervals, by hop count

Accuracy is token F1 (MuSiQue, 2Wiki) or Hits@1 (MetaQA). The three right-hand columns are paired bootstrap
intervals over questions; **DAG − ReAct is the primary contrast**. MuSiQue groups are its hop counts, 2Wiki groups
its question types (comparison and bridge-comparison have independent sub-questions), MetaQA is all 3-hop.

In [8]:
SUMMARY = {}
CONTRASTS = (("react", "dag"), ("list", "dag"), ("vector", "dag"))
ci = lambda t: f"{t[0] * 100:+.1f} [{t[1] * 100:+.1f}, {t[2] * 100:+.1f}]"

def table(label, rows, metric="acc", groups=True):
    print(f"\n{label}  (n = {len(rows)}; {'Hits@1' if rows and rows[0]['bench'] == 'metaqa' else 'token F1'})")
    print(f"{'group':<20}{'n':>5}" + "".join(f"{NAMES[m]:>14}" for m in METHODS) +
          "".join(f"{NAMES[b].split()[0] + ' vs ' + NAMES[a].split()[0]:>26}" for a, b in CONTRASTS))
    gs = (sorted({r["group"] for r in rows}) if groups else []) + ["ALL"]
    for g in gs:
        rr = [r for r in rows if g == "ALL" or r["group"] == g]
        if not rr:
            continue
        means = {m: np.mean([r[m][metric] for r in rr]) for m in METHODS}
        diffs = {(a, b): paired_boot([r[a][metric] for r in rr], [r[b][metric] for r in rr]) for a, b in CONTRASTS}
        SUMMARY[(label, g, metric)] = (len(rr), means, diffs)
        print(f"{g:<20}{len(rr):>5}" + "".join(f"{means[m]:>14.1%}" for m in METHODS) +
              "".join(f"{ci(diffs[c]):>26}" for c in CONTRASTS))

for b in BENCH:
    table(b, [r for r in SC if r["bench"] == b])
table("musique+2wiki", [r for r in SC if r["bench"] != "metaqa"], groups=False)
print("\nExact match (MuSiQue + 2Wiki):", ", ".join(
    f"{NAMES[m]} {np.mean([r[m]['em'] for r in SC if r['bench'] != 'metaqa']):.1%}" for m in METHODS))
print("Set F1 (MetaQA):", ", ".join(f"{NAMES[m]} {np.mean([r[m]['f1'] for r in SC if r['bench'] == 'metaqa']):.1%}"
                                   for m in METHODS))
for lab in ("musique+2wiki", "metaqa"):
    n, _, d = SUMMARY[(lab, "ALL", "acc")]
    print(f"Interval half-width of DAG - ReAct on {lab}: ±{(d[('react', 'dag')][2] - d[('react', 'dag')][1]) * 50:.1f} "
          f"points (PLAN §5b rule 3 asks for < ±5 in the full run)")


musique  (n = 1000; token F1)
group                   n    vector RAG         ReAct  list decomp.   DAG planner              DAG vs ReAct               DAG vs list             DAG vs vector
2-hop                 518         43.7%         56.5%         55.9%         58.6%         +2.0 [-1.5, +5.5]         +2.7 [-0.1, +5.5]      +14.8 [+11.0, +18.8]
3-hop                 316         36.6%         50.2%         45.2%         43.2%        -7.0 [-12.0, -2.0]         -2.1 [-6.4, +2.2]        +6.6 [+1.1, +12.3]
4-hop                 166         24.3%         38.0%         23.1%         25.1%       -12.9 [-19.9, -6.0]         +2.0 [-3.5, +7.7]         +0.8 [-5.4, +7.0]
ALL                  1000         38.2%         51.5%         47.1%         48.1%         -3.3 [-6.0, -0.5]         +1.1 [-1.1, +3.3]        +9.9 [+7.0, +12.8]

2wikimultihopqa  (n = 1000; token F1)
group                   n    vector RAG         ReAct  list decomp.   DAG planner              DAG vs ReAct               DAG vs l

ALL                  1000         51.2%         73.3%         58.1%         60.3%      -13.0 [-15.8, -10.3]         +2.2 [+0.1, +4.2]        +9.1 [+5.6, +12.4]

metaqa  (n = 300; Hits@1)
group                   n    vector RAG         ReAct  list decomp.   DAG planner              DAG vs ReAct               DAG vs list             DAG vs vector
3-hop                 300         12.3%         48.0%         28.3%         28.3%      -19.7 [-26.0, -13.0]         +0.0 [-4.7, +4.7]      +16.0 [+10.3, +21.7]
ALL                   300         12.3%         48.0%         28.3%         28.3%      -19.7 [-26.0, -13.0]         +0.0 [-4.7, +4.7]      +16.0 [+10.3, +21.7]

musique+2wiki  (n = 2000; token F1)
group                   n    vector RAG         ReAct  list decomp.   DAG planner              DAG vs ReAct               DAG vs list             DAG vs vector
ALL                  2000         44.7%         62.4%         52.6%         54.2%        -8.2 [-10.1, -6.2]         +1.6 [+0.1, +3.1]   

## Cost: LLM calls, tokens and sequential rounds per question

Means per question. USD is at the Batch price (`smash_batch.PRICE`), for comparability across conditions; the
vector RAG tokens are those of the reused answers. *Rounds* is the number of LLM calls on the critical path
(sequential rounds): ReAct's steps, the list's plan + one per sub-question + final, the DAG's plan + its depth + join.

In [9]:
EFF = {}
print(f"{'benchmark':<18}{'condition':<15}{'calls':>7}{'in tok':>9}{'out tok':>9}{'USD/1k q':>10}{'rounds':>8}"
      f"{'accuracy':>10}")
for b in BENCH + ["ALL"]:
    rr = [r for r in SC if b == "ALL" or r["bench"] == b]
    for m in METHODS:
        e = {k: np.mean([r[m][k] for r in rr]) for k in ("calls", "tin", "tout", "usd", "rounds", "acc")}
        EFF[(b, m)] = e
        print(f"{b:<18}{NAMES[m]:<15}{e['calls']:>7.2f}{e['tin']:>9,.0f}{e['tout']:>9,.0f}{e['usd'] * 1000:>10.2f}"
              f"{e['rounds']:>8.2f}{e['acc']:>10.1%}")
for m in ("list", "dag"):
    print(f"{NAMES[m]} / ReAct (all benchmarks): calls x{EFF[('ALL', m)]['calls'] / EFF[('ALL', 'react')]['calls']:.2f}, "
          f"tokens x{(EFF[('ALL', m)]['tin'] + EFF[('ALL', m)]['tout']) / (EFF[('ALL', 'react')]['tin'] + EFF[('ALL', 'react')]['tout']):.2f}, "
          f"rounds x{EFF[('ALL', m)]['rounds'] / EFF[('ALL', 'react')]['rounds']:.2f}")

benchmark         condition        calls   in tok  out tok  USD/1k q  rounds  accuracy
musique           vector RAG        1.00      787       21      0.04    1.00     38.2%
musique           ReAct             3.83    5,590      123      0.31    3.83     51.5%
musique           list decomp.      4.54    3,314       71      0.18    4.54     47.1%
musique           DAG planner       4.45    3,337      100      0.19    4.16     48.1%
2wikimultihopqa   vector RAG        1.00      821       22      0.05    1.00     51.2%
2wikimultihopqa   ReAct             3.15    3,415      105      0.20    3.15     73.3%
2wikimultihopqa   list decomp.      4.36    3,196       72      0.18    4.36     58.1%
2wikimultihopqa   DAG planner       4.33    3,308       96      0.19    3.65     60.3%
metaqa            vector RAG        1.00    1,657       15      0.09    1.00     12.3%
metaqa            ReAct             5.99   12,880      179      0.69    5.99     48.0%
metaqa            list decomp.      4.32   

## Does the planned DAG match MuSiQue's gold decomposition?

MuSiQue gives each question its gold chain of sub-questions, with `#k` marking which earlier answer a step uses.
For each plan: does it have the gold number of hops, and (if so) the gold dependency structure (the same directed
graph up to node order)? Shown for the DAG planner and the list (whose structure is the chain implied by its `#k`).

In [10]:
def gold_graph(q):
    G = nx.DiGraph(); G.add_nodes_from(range(len(q["question_decomposition"])))
    for i, s in enumerate(q["question_decomposition"]):
        G.add_edges_from((int(k) - 1, i) for k in REF.findall(s["question"]) if 0 < int(k) <= i)
    return G

def plan_graph(nodes, use_refs):
    G = nx.DiGraph(); G.add_nodes_from(range(len(nodes)))
    for i, n in enumerate(nodes):
        G.add_edges_from((j, i) for j in (n["refs"].values() if use_refs else n["deps"]))
    return G

HOPS = {}
print(f"{'gold hops':<12}{'n':>5}{'DAG: same #hops':>17}{'same graph':>12}{'mean #nodes':>13}"
      f"{'list: same #hops':>18}{'same graph':>12}{'mean #nodes':>13}")
mq = DATA["musique"]["questions"]
for h in sorted({len(q["question_decomposition"]) for q in mq}) + ["ALL"]:
    qs = [q for q in mq if h == "ALL" or len(q["question_decomposition"]) == h]
    row = {}
    for m, M, refs in (("dag", DAG, False), ("list", LIST, True)):
        nodes = [M[("musique", q["qid"])]["nodes"] for q in qs]
        same_n = [len(n) == len(q["question_decomposition"]) for n, q in zip(nodes, qs)]
        same_g = [nx.is_isomorphic(plan_graph(n, refs), gold_graph(q)) for n, q in zip(nodes, qs)]
        row[m] = (np.mean(same_n), np.mean(same_g), np.mean([len(n) for n in nodes]))
    HOPS[h] = (len(qs), row)
    print(f"{str(h):<12}{len(qs):>5}" + "".join(f"{row[m][0]:>17.0%}{row[m][1]:>12.0%}{row[m][2]:>13.2f}" for m in ("dag", "list")))

# Accuracy of the DAG planner when its plan has the gold hop count vs. when it does not
same = [len(DAG[("musique", q["qid"])]["nodes"]) == len(q["question_decomposition"]) for q in mq]
f1d = {q["qid"]: r["dag"]["f1"] for q, r in zip(mq, [r for r in SC if r["bench"] == "musique"])}
print("\nDAG F1 on MuSiQue when the plan has the gold hop count: "
      f"{np.mean([f1d[q['qid']] for q, s in zip(mq, same) if s]):.1%} (n={sum(same)}); otherwise "
      f"{np.mean([f1d[q['qid']] for q, s in zip(mq, same) if not s]) if not all(same) else float('nan'):.1%} (n={len(same) - sum(same)})")

gold hops       n  DAG: same #hops  same graph  mean #nodes  list: same #hops  same graph  mean #nodes
2             518              88%         79%         1.93              86%         69%         2.02
3             316              62%         44%         2.78              67%         46%         2.91
4             166              39%         30%         3.45              48%         28%         3.49
ALL          1000              72%         60%         2.45              74%         55%         2.54

DAG F1 on MuSiQue when the plan has the gold hop count: 54.0% (n=720); otherwise 33.1% (n=280)


## Wall-clock latency on a synchronous subset

The first 100 MuSiQue and 100 2Wiki questions (test: 5 + 5) are run again, **one question at a time**, with
synchronous calls at the standard tier: the wall-clock time from the question to the final answer, including the
query embeddings and searches. The DAG's parallel nodes run concurrently (thread pool), the list's one after another.
These calls are cached under `nb25_lat_*`; the measured times are saved in `nb25_latency.json` and replayed. They are
fresh calls, so their answers can differ slightly from the Batch run's; accuracy on this subset is reported too.

In [11]:
LAT_PATH = f"{llm.cache}/nb25_latency.json"
LAT = json.load(open(LAT_PATH)) if os.path.exists(LAT_PATH) else {}

class LiveRunner(Runner):
    """Synchronous, one question at a time. Query embeddings are sent live (not cached), so their time counts."""
    def __init__(self, llm, prefix):
        super().__init__(llm, prefix); self.emb_tokens = 0
    def embed(self, name, texts):
        if not texts:
            return np.zeros((0, 1536), np.float32)
        r = self.llm.client.embeddings.create(model=EMBED_MODEL, input=list(texts))
        self.emb_tokens += r.usage.total_tokens
        v = np.array([e.embedding for e in r.data], np.float32)
        return v / np.linalg.norm(v, axis=1, keepdims=True)

def run_vector_live(R, eps, tag):
    ep = eps[0]; b = ep["bench"]
    ids = search(R, tag, [(b, ep["question"])])[0]
    ctx = "\n\n".join(f"[{n + 1}] {DATA[b]['passages'][i]}" for n, i in enumerate(ids))
    r = R.chat(tag, {ek(ep): LLM.body([{"role": "system", "content": NB11_SYS},
                                       {"role": "user", "content": f"{ctx}\n\nQuestion: {ep['question']}"}], max_tokens=300)})[ek(ep)]
    acc(ep, r); ep["rounds"] += 1; ep["text"] = r.get("text") or ""
    return eps

LAT_FN = {"vector": run_vector_live, "react": run_react,
          "list": lambda R, e, t: run_planner(R, e, "list", t), "dag": lambda R, e, t: run_planner(R, e, "dag", t)}
LAT_Q = [(b, qi) for b in ("musique", "2wikimultihopqa") for qi in range(min(N_LAT, len(DATA[b]["questions"])))]
todo = [(b, qi, m) for b, qi in LAT_Q for m in METHODS if f"{m}|{b}|{DATA[b]['questions'][qi]['qid']}" not in LAT]
if todo and MODE == "replay":
    raise RuntimeError(f"{len(todo)} latency measurements missing - run with SMASH_MODE=test or batch")
if todo:
    lat_llm = LLM(llm.cache, mode="test", workers=16)          # synchronous at standard tier, also in batch mode
    R = LiveRunner(lat_llm, "nb25_lat_")
    print(f"measuring {len(todo)} episodes, one at a time ...", flush=True)
    for n, (b, qi, m) in enumerate(todo):
        ep = new_ep(b, qi); e0 = R.emb_tokens
        t0 = time.perf_counter()
        LAT_FN[m](R, [ep], m)
        sec = time.perf_counter() - t0
        LAT[f"{m}|{b}|{ep['qid']}"] = {"sec": sec, "calls": ep["calls"], "rounds": ep["rounds"], "tin": ep["tin"],
                                       "tout": ep["tout"], "emb_tokens": R.emb_tokens - e0,
                                       "acc": score(b, DATA[b]["questions"][qi], ep["text"])["acc"]}
        json.dump(LAT, open(LAT_PATH, "w"))                    # saved after every episode: an interrupted run resumes
        if (n + 1) % 40 == 0:
            print(f"  {n + 1}/{len(todo)}", flush=True)

print(f"{'condition':<15}{'n':>5}{'median s':>10}{'mean s':>9}{'p90 s':>8}{'rounds':>8}{'calls':>7}{'accuracy':>10}")
LATS = {}
for m in METHODS:
    rows = [LAT[f"{m}|{b}|{DATA[b]['questions'][qi]['qid']}"] for b, qi in LAT_Q]
    s = np.array([r["sec"] for r in rows])
    LATS[m] = s
    print(f"{NAMES[m]:<15}{len(s):>5}{np.median(s):>10.2f}{s.mean():>9.2f}{np.percentile(s, 90):>8.2f}"
          f"{np.mean([r['rounds'] for r in rows]):>8.2f}{np.mean([r['calls'] for r in rows]):>7.2f}"
          f"{np.mean([r['acc'] for r in rows]):>10.1%}")
for a, b in (("react", "dag"), ("list", "dag")):
    d = paired_boot(LATS[a], LATS[b])
    print(f"latency {NAMES[b]} - {NAMES[a]}: {d[0]:+.2f} s [{d[1]:+.2f}, {d[2]:+.2f}]; "
          f"median speed-up x{np.median(LATS[a]) / np.median(LATS[b]):.2f}")
for b in ("musique", "2wikimultihopqa"):
    qs = DATA[b]["questions"]
    for g in sorted({qs[qi]["group"] for bb, qi in LAT_Q if bb == b}):
        sel = [qi for bb, qi in LAT_Q if bb == b and qs[qi]["group"] == g]
        print(f"  {b} {g:<18} n={len(sel):<4}" + "  ".join(
            f"{NAMES[m]} {np.median([LAT[f'{m}|{b}|{qs[qi]['qid']}']['sec'] for qi in sel]):.1f}s" for m in METHODS))

measuring 800 episodes, one at a time ...


[nb25_lat_vector] 1 new requests -> test (0 cached)


[nb25_lat_react_s1] 1 new requests -> test (0 cached)


[nb25_lat_react_s2] 1 new requests -> test (0 cached)


[nb25_lat_react_s3] 1 new requests -> test (0 cached)


[nb25_lat_react_s4] 1 new requests -> test (0 cached)


... (2,608 outputs left out)

[nb25_lat_dag_w1] 1 new requests -> test (0 cached)


[nb25_lat_dag_final] 1 new requests -> test (0 cached)


  800/800


condition          n  median s   mean s   p90 s  rounds  calls  accuracy
vector RAG       200      1.50     1.59    1.94    1.00   1.00     43.0%
ReAct            200      4.68     5.63    9.25    3.52   3.52     60.1%
list decomp.     200      5.93     6.45    8.70    4.49   4.49     56.7%
DAG planner      200      6.28     6.64    9.37    3.97   4.46     53.2%
latency DAG planner - ReAct: +1.01 s [+0.58, +1.39]; median speed-up x0.75
latency DAG planner - list decomp.: +0.20 s [-0.08, +0.47]; median speed-up x0.94
  musique 2-hop              n=48  vector RAG 1.5s  ReAct 4.4s  list decomp. 5.6s  DAG planner 5.9s
  musique 3-hop              n=30  vector RAG 1.5s  ReAct 7.0s  list decomp. 7.3s  DAG planner 6.8s
  musique 4-hop              n=22  vector RAG 1.4s  ReAct 8.1s  list decomp. 8.1s  DAG planner 8.6s
  2wikimultihopqa bridge_comparison  n=23  vector RAG 1.6s  ReAct 5.9s  list decomp. 7.8s  DAG planner 6.8s
  2wikimultihopqa comparison         n=25  vector RAG 1.5s  ReAct 4.4s

## What this run cost

Only this notebook's stages are paid here (the passage and article indexes and the vector answers are reused). The
forecast scales the per-question cost of this run to the full n, at the Batch price for the main conditions and the
standard price for the latency subset.

In [12]:
LEDGER = llm.ledger()
lat_emb = sum(v["emb_tokens"] for v in LAT.values())
print(f"{'stage':<26}{'calls':>8}{'input tok':>13}{'output tok':>12}{'USD':>8}{'via Batch':>11}")
for s, a in LEDGER.items():
    print(f"{s:<26}{a['calls']:>8,}{a['in']:>13,}{a['out']:>12,}{a['usd']:>8.3f}{a['batch_share']:>11.0%}")
total = sum(a["usd"] for a in LEDGER.values()) + lat_emb * EMBED_PRICE / 1e6
print(f"{'(latency query embeddings)':<26}{'':>8}{lat_emb:>13,}{'':>12}{lat_emb * EMBED_PRICE / 1e6:>8.3f}")
print(f"{'TOTAL':<26}{sum(a['calls'] for a in LEDGER.values()):>8,}{'':>25}{total:>8.3f}")

# Forecast of the full run from this run's tokens per question
n_full = {"musique": 1000, "2wikimultihopqa": 1000, "metaqa": 300}
main = sum(n_full[r["bench"]] / sum(1 for x in SC if x["bench"] == r["bench"]) *
           sum(r[m]["usd"] for m in ("react", "list", "dag")) for r in SC)
calls = sum(n_full[r["bench"]] / sum(1 for x in SC if x["bench"] == r["bench"]) *
            sum(r[m]["calls"] for m in ("react", "list", "dag")) for r in SC)
lat_q = [v for v in LAT.values()]
lat_usd = 200 / max(len(LAT_Q), 1) * sum((v["tin"] * PRICE["standard"][0] + v["tout"] * PRICE["standard"][1]) / 1e6 for v in lat_q)
lat_calls = 200 / max(len(LAT_Q), 1) * sum(v["calls"] for v in lat_q)
print(f"\nForecast, full run: main conditions {calls:,.0f} calls, ${main:.2f} (Batch); latency subset {lat_calls:,.0f} calls, "
      f"${lat_usd:.2f} (standard); total ${main + lat_usd:.2f} (cost_forecast.py: 37,500 calls, $3.24)")

stage                        calls    input tok  output tok     USD  via Batch
nb25_dag_final               2,300    2,356,633      32,178   0.126       100%
nb25_dag_plan                2,300      759,623     160,936   0.078       100%
nb25_dag_w1                  3,092    2,411,727      21,766   0.126       100%
nb25_dag_w2                  1,999    1,848,599      13,248   0.096       100%
nb25_dag_w3                    364      349,134       2,132   0.018       100%
nb25_dag_w4                     52       47,820         228   0.002       100%
nb25_dag_w5                      5        4,376          23   0.000       100%
nb25_lat_dag_final             200      203,863       2,863   0.022         0%
nb25_lat_dag_plan              200       65,947      14,070   0.014         0%
nb25_lat_dag_w1                273      202,444       1,855   0.021         0%
nb25_lat_dag_w2                179      167,852       1,046   0.017         0%
nb25_lat_dag_w3                 29       27,438     


Forecast, full run: main conditions 29,095 calls, $1.58 (Batch); latency subset 2,693 calls, $0.27 (standard); total $1.85 (cost_forecast.py: 37,500 calls, $3.24)


## Limitations

- **One configuration.** One model (`gpt-6-luna`, no reasoning), one prompt per condition, top-5 search, at most 5
  searches or 5 sub-questions; each configuration run once. Prompts were fixed before the run, not tuned.
- **ReAct is sequential by design.** Parallel tool calls are disabled to keep the classic ReAct baseline that
  LLMCompiler compares against; a modern function-calling agent that may issue several searches per step would
  narrow the latency gap.
- **Errors propagate in plans.** A wrong or `unknown` sub-answer is substituted into later sub-questions; the
  final/join call sees the top passages and can sometimes recover, but no re-planning is done (LLMCompiler's
  re-planning loop is not implemented).
- **Set-valued hops on MetaQA.** A sub-question whose answer is a set (all films of a director) gets one search of 5
  articles; all agent conditions are capped by this, and the text lacks tags and IMDb facts (Notebook 1.2b).
- **Latency** is one synchronous measurement per question on the shared API; it depends on load and network. The
  sequential-rounds count, available for every question, is the load-independent proxy.
- **Hop-count match** compares node counts and dependency graphs only; it does not check that a planned
  sub-question means the same as the gold one.

**References.** Kim et al., *An LLM Compiler for Parallel Function Calling* (LLMCompiler), ICML 2024 · Yao et al.,
*ReAct*, ICLR 2023 · Besta et al., *Graph of Thoughts*, AAAI 2024 · Trivedi et al., *MuSiQue*, TACL 2022 · Ho et
al., *2WikiMultiHopQA*, COLING 2020 · Zhang et al., *MetaQA*, AAAI 2018.

In [13]:
print("=" * 120)
print(f"RESULTS SUMMARY — Notebook 4.1: planning with a graph (task 4.1), {MODE} run, gpt-6-luna (no reasoning)")
print("=" * 120)
print(f"{'benchmark':<16}{'n':>5}" + "".join(f"{NAMES[m]:>14}" for m in METHODS) +
      f"{'DAG - ReAct [95% CI]':>26}{'DAG - list':>24}{'DAG - vector':>24}")
verdict = {}
for lab in BENCH + ["musique+2wiki"]:
    n, means, diffs = SUMMARY[(lab, "ALL", "acc")]
    print(f"{lab:<16}{n:>5}" + "".join(f"{means[m]:>14.1%}" for m in METHODS) +
          "".join(f"{ci(diffs[c]):>{26 if c == ('react', 'dag') else 24}}" for c in CONTRASTS))
    d = diffs[("react", "dag")]
    verdict[lab] = "DAG better" if d[1] > 0 else "DAG worse" if d[2] < 0 else "no resolvable difference"
print("(accuracy = token F1 on MuSiQue/2Wiki, Hits@1 on MetaQA)")
print("\nPer question, all benchmarks:  " + " | ".join(
    f"{NAMES[m]}: {EFF[('ALL', m)]['calls']:.1f} calls, {EFF[('ALL', m)]['tin'] + EFF[('ALL', m)]['tout']:,.0f} tok, "
    f"{EFF[('ALL', m)]['rounds']:.1f} rounds" for m in METHODS))
print("Latency subset (median s):  " + " | ".join(f"{NAMES[m]} {np.median(LATS[m]):.1f}" for m in METHODS) +
      f"  (n = {len(LAT_Q)})")
print(f"MuSiQue plans with the gold hop count: DAG {HOPS['ALL'][1]['dag'][0]:.0%} (same graph {HOPS['ALL'][1]['dag'][1]:.0%}), "
      f"list {HOPS['ALL'][1]['list'][0]:.0%} (same graph {HOPS['ALL'][1]['list'][1]:.0%})")
print("\nVerdict, DAG planner vs ReAct on the primary metric:", "; ".join(f"{k}: {v}" for k, v in verdict.items()))
print("\nLecture message (Task 4.1): a plan written as a graph of sub-questions runs independent hops in parallel and"
      "\nkeeps no growing transcript; whether that costs accuracy against a step-by-step agent is what the verdict says.")

RESULTS SUMMARY — Notebook 4.1: planning with a graph (task 4.1), batch run, gpt-6-luna (no reasoning)
benchmark           n    vector RAG         ReAct  list decomp.   DAG planner      DAG - ReAct [95% CI]              DAG - list            DAG - vector
musique          1000         38.2%         51.5%         47.1%         48.1%         -3.3 [-6.0, -0.5]       +1.1 [-1.1, +3.3]      +9.9 [+7.0, +12.8]
2wikimultihopqa  1000         51.2%         73.3%         58.1%         60.3%      -13.0 [-15.8, -10.3]       +2.2 [+0.1, +4.2]      +9.1 [+5.6, +12.4]
metaqa            300         12.3%         48.0%         28.3%         28.3%      -19.7 [-26.0, -13.0]       +0.0 [-4.7, +4.7]    +16.0 [+10.3, +21.7]
musique+2wiki    2000         44.7%         62.4%         52.6%         54.2%        -8.2 [-10.1, -6.2]       +1.6 [+0.1, +3.1]      +9.5 [+7.2, +11.6]
(accuracy = token F1 on MuSiQue/2Wiki, Hits@1 on MetaQA)

Per question, all benchmarks:  vector RAG: 1.0 calls, 936 tok, 1.0 rounds | ReA